# Super-Cortex // PyTorch Geometric (PyG) GNN Node Classification
This notebook fetches the live 3D neural knowledge manifold from Super-Cortex,
converts it into a `torch_geometric.data.Data` graph tensor, and trains a Graph Convolutional Network (GCN).


In [ ]:
# 1. Install & import required libraries
# pip install torch torch_geometric httpx polars matplotlib
import httpx
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

CORTEX_HOST = 'http://127.0.0.1:4174'
SCOPE = 'all'


In [ ]:
# 2. Ingest graph tensors directly from Super-Cortex REST API
resp = httpx.get(f'{CORTEX_HOST}/api/export/pyg', params={'scope': SCOPE})
data_spec = resp.json()

print(f"Ingested {data_spec['num_nodes']} nodes and {data_spec['num_edges']} edges.")

# Construct PyG Data object
edge_index = torch.tensor(data_spec['edge_index'], dtype=torch.long)
y = torch.tensor(data_spec['y'], dtype=torch.long)
num_classes = len(data_spec['node_types'])

# Initialize node feature matrix X (one-hot or degree + degree centralities)
num_nodes = data_spec['num_nodes']
x = torch.eye(num_nodes) if num_nodes <= 1000 else torch.randn(num_nodes, 32)

pyg_graph = Data(x=x, edge_index=edge_index, y=y)
print(pyg_graph)


In [ ]:
# 3. Define 2-layer Graph Convolutional Network (GCN)
class CortexGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

model = CortexGCN(x.size(1), 16, num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Training loop (30 epochs)
model.train()
for epoch in range(1, 31):
    optimizer.zero_grad()
    out = model(pyg_graph.x, pyg_graph.edge_index)
    loss = F.nll_loss(out, pyg_graph.y)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f'Epoch {epoch:02d} | Loss: {loss.item():.4f}')
